# 7.9 Feedforward Networks for Tabular Data

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook applies dense neural networks to structured feature tables and shows where they fit relative to simpler baselines.


## 7.9.1 Dense networks on structured features

### Why It Matters
Tabular data does not have the spatial or sequential structure of images and text, so the default neural architecture is usually a feedforward network.


In [ ]:
import torch
from torch import nn

X = torch.randn(32, 7)
model = nn.Sequential(nn.Linear(7, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1))
out = model(X)
{"input_shape": list(X.shape), "output_shape": list(out.shape), "parameter_count": sum(p.numel() for p in model.parameters())}


### Interpretation
The architecture is simple because the structure in tabular data usually lives in the feature engineering rather than in the input geometry.


## 7.9.2 Embeddings for categorical variables

### Why It Matters
High-cardinality categories are often represented more efficiently with learned embeddings than with huge one-hot vectors.


In [ ]:
import torch
from torch import nn

categories = torch.tensor([0, 3, 1, 4, 2])
embed = nn.Embedding(num_embeddings=5, embedding_dim=3)
vectors = embed(categories)
{"embedding_shape": list(vectors.shape), "category_3_vector": vectors[1].detach().round(decimals=3).tolist()}


### Interpretation
An embedding lets the model learn a compact dense representation of each category instead of allocating one feature per level.


## 7.9.3 Regression versus classification heads

### Why It Matters
The hidden trunk can stay similar while only the prediction head changes between regression and classification tasks.


In [ ]:
import torch
from torch import nn

trunk = nn.Sequential(nn.Linear(6, 10), nn.ReLU())
X = torch.randn(4, 6)
shared = trunk(X)
regression_head = nn.Linear(10, 1)
classification_head = nn.Sequential(nn.Linear(10, 1), nn.Sigmoid())
{
    "shared_representation_shape": list(shared.shape),
    "regression_output": regression_head(shared).flatten().round(decimals=3).tolist(),
    "classification_output": classification_head(shared).flatten().round(decimals=3).tolist(),
}


### Interpretation
The same feature extractor can support different output semantics depending on the downstream task.


## 7.9.4 Handling mixed numeric and categorical data

### Why It Matters
A common tabular pattern is to concatenate numeric features with learned embeddings for categoricals before the dense network.


In [ ]:
import torch
from torch import nn

numeric = torch.randn(5, 3)
category = torch.tensor([0, 1, 2, 1, 0])
embed = nn.Embedding(3, 2)
combined = torch.cat([numeric, embed(category)], dim=1)
model = nn.Sequential(nn.Linear(5, 8), nn.ReLU(), nn.Linear(8, 1))
out = model(combined)
{"combined_feature_shape": list(combined.shape), "output_shape": list(out.shape)}


### Interpretation
This pattern is common in recommendation, retail, and transactional models where tables mix several data types.


## 7.9.5 Calibration concerns

### Why It Matters
A classifier can be accurate while still producing overconfident probabilities. Brier score is one compact way to inspect probabilistic calibration.


In [ ]:
import torch
from torch import nn
from sklearn.metrics import brier_score_loss

torch.manual_seed(31)
X = torch.randn(160, 4)
y = ((X[:, 0] + X[:, 1]) > 0).int().numpy()
model = nn.Sequential(nn.Linear(4, 12), nn.ReLU(), nn.Linear(12, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.03)
y_t = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)
loss_fn = nn.BCELoss()
for _ in range(120):
    opt.zero_grad()
    loss = loss_fn(model(X), y_t)
    loss.backward()
    opt.step()
with torch.no_grad():
    probs = model(X).numpy().ravel()
{"brier_score": round(float(brier_score_loss(y, probs)), 4), "avg_predicted_probability": round(float(probs.mean()), 4)}


### Interpretation
Probabilities are a modeling product of their own, not just a side effect of classification.


## 7.9.6 Tabular baseline versus MLP

### Why It Matters
A neural network on tabular data should still justify itself against a simple linear baseline.


In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=400, n_features=10, n_informative=6, n_redundant=0, random_state=32)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=32)
baseline = LogisticRegression(max_iter=2000).fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))

Xt = torch.tensor(X_train, dtype=torch.float32)
yt = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
Xv = torch.tensor(X_test, dtype=torch.float32)
model = nn.Sequential(nn.Linear(10, 20), nn.ReLU(), nn.Linear(20, 1), nn.Sigmoid())
opt = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = nn.BCELoss()
for _ in range(200):
    opt.zero_grad()
    loss = loss_fn(model(Xt), yt)
    loss.backward()
    opt.step()
with torch.no_grad():
    mlp_preds = (model(Xv) > 0.5).int().numpy().ravel()
{"logistic_accuracy": round(float(baseline_acc), 3), "mlp_accuracy": round(float((mlp_preds == y_test).mean()), 3)}


### Interpretation
The comparison forces you to ask whether the network is adding predictive value or just extra complexity.
